In [10]:
import pandas as pd
from scipy.stats import ttest_ind
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv("D:/Habuild_Olist_CaseStudy_Aditya/notebook/clean_data.csv")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,has_review,delivery_days,approval_hours,shipping_days,is_late,purchase_year,purchase_month,purchase_day,purchase_hour,purchase_day_name
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,True,8.0,0.178333,2.0,False,2017,10,2,10,Monday
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,True,13.0,30.713889,1.0,False,2018,7,24,20,Tuesday
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,True,9.0,0.276111,0.0,False,2018,8,8,8,Wednesday
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,True,13.0,0.298056,3.0,False,2017,11,18,19,Saturday
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,True,2.0,1.030556,0.0,False,2018,2,13,21,Tuesday


In [4]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_value', 'payment_installments',
       'payment_type', 'price', 'freight_value', 'total_items', 'review_score',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state', 'is_delivered', 'has_review',
       'delivery_days', 'approval_hours', 'shipping_days', 'is_late',
       'purchase_year', 'purchase_month', 'purchase_day', 'purchase_hour',
       'purchase_day_name'],
      dtype='str')

In [ ]:
# map the values
# df['is_late'] = df['is_late'].map({False: 0, True: 1})

le = LabelEncoder()
df['is_late'] = le.fit_transform(df['is_late'])

df['is_late'][5:10]

5    0
6    0
7    0
8    0
9    0
Name: is_late, dtype: int64

In [14]:
df['is_late'].value_counts()

is_late
0    91614
1     7827
Name: count, dtype: int64

### Hypothesis Testing

**Problem**	                     **Test**
```
2 groups average compare	    T-Test
Categories compare	            Chi-Square
Numeric relationship	        Correlation
3+ groups compare	            ANOVA
```

In [ ]:
# Null Hypothesis (H0) = Delivery delays do NOT affect review scores
# Alternative Hypothesis (H1) = Delivery delays DO affect review scores.


late_delivery = df[df['is_late'] == 1]['review_score']

on_time_delivery = df[df['is_late'] == 0]['review_score']


# t-test because tow groupes are there
t_stat, p_value = ttest_ind(late_delivery, on_time_delivery,nan_policy='omit')


print("T-Statistic:", t_stat)
print("P-Value:", p_value)

T-Statistic: -108.93232066030252
P-Value: 0.0


I reject the null hypothesis because the p-value is exactly 0.0, which falls well below the standard 0.05 significance level. The extremely large negative T-statistic (-108.93) proves that the sample mean is vastly lower than the hypothesized mean, guaranteeing these findings did not happen by random chance. Ultimately, this confirms that late deliveries significantly reduce customer review scores, indicating that delivery performance strongly impacts customer satisfaction and requires operational improvement.

In [ ]:
# h0 = Installment users and non-installment users spend the same amount.
# h1 = Installment users spend more money.

single_payment = df[df['payment_installments'] == 1]['payment_value']

installment_payment = df[df['payment_installments'] > 1]['payment_value']


t_stat, p_value = ttest_ind(
    single_payment,
    installment_payment
)

print("T-Statistic:", t_stat)
print("P-Value:", p_value)

T-Statistic: -55.99263447314089
P-Value: 0.0


Know we reject the null hypothesis because this test show the significant difference in spending behavior between single-payment and installment-payment customers (p < 0.05). Customers using installment payments tend to spend significantly more money compared to customers paying in a single transaction. so we learn through this to provie the flexible payment options to encourage customers to purchase higher-value products.

### Chi-Square Test

In [ ]:
# h0 = Payment type and order status are independent.
# h1 = Payment type and order status are related.

table = pd.crosstab(
    df['payment_type'],
    df['order_status']
)

table

order_status,approved,canceled,created,delivered,invoiced,processing,shipped,unavailable
payment_type,,,,,,,,
boleto,0,95,2,19191,67,70,209,150
credit_card,2,440,3,73942,237,222,847,440
debit_card,0,7,0,1484,6,2,22,6
not_defined,0,3,0,0,0,0,0,0
voucher,0,80,0,1861,4,7,29,13


In [30]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-Square Value:", chi2)
print("P-Value:", p)

Chi-Square Value: 872.0018935757879
P-Value: 1.5113522711439135e-165


I reject the null hypothesis because the p-value is exactly 0.0 (1.51e-165), which is vastly lower than the standard significance level of 0.05. The Chi-Square value is extremely large (872.00), which guarantees that the relationship between payment type and order status did not happen by random chance. This proves that different payment methods strongly impact order outcomes like successful deliveries, cancellations, or processing issues. The company should definitely analyze which payment methods have higher failure rates to improve transaction efficiency and customer experience